# 🏦 Santander Product Recommendation — Pipeline xử lý dữ liệu

Notebook này xây dựng toàn bộ pipeline cho bài toán **dự đoán sản phẩm ngân hàng mới mà khách hàng sẽ mua thêm** (Kaggle *Santander Product Recommendation*), gồm 5 giai đoạn:

1. **Data Preparation** — nạp dữ liệu train/test từ file nén gốc.
2. **Data Preprocessing** — tối ưu bộ nhớ, xử lý giá trị NULL, mã hoá biến phân loại.
3. **Feature Engineering** — loại bỏ đặc trưng/sản phẩm không hữu ích, tạo đặc trưng thời gian và đặc trưng lag.
4. **Chuẩn bị tập Train/Test** — chuyển về dạng long-format theo từng sản phẩm để train multi-class.
5. **Huấn luyện & Dự đoán** — tìm siêu tham số, train XGBoost, xuất top-7 sản phẩm dự đoán cho mỗi khách hàng (theo chuẩn đánh giá MAP@7 của cuộc thi).

## 1. Data Preparation

Import các thư viện cần dùng, giải nén 3 file dữ liệu gốc (`train_ver2`, `test_ver2`, `sample_submission`) từ thư mục input của Kaggle ra thư mục làm việc, sau đó đọc `train_ver2.csv` và `test_ver2.csv` vào `pandas.DataFrame`.

In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ctypes
import gc
from tqdm import tqdm
import pickle
from scipy import stats
import collections
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Định nghĩa đường dẫn
input_dir = "/kaggle/input/santander-product-recommendation/"
output_dir = "/kaggle/working/"

# Danh sách file cần giải nén
zip_files = [
    "train_ver2.csv.zip",
    "test_ver2.csv.zip",
    "sample_submission.csv.zip"
]

# Giải nén từng file
for zip_file in zip_files:
    zip_path = os.path.join(input_dir, zip_file)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(output_dir)

In [ ]:
df = pd.read_csv(os.path.join(output_dir, "train_ver2.csv"), low_memory=False)

In [ ]:
df_test = pd.read_csv(os.path.join(output_dir, "test_ver2.csv"), low_memory=False)

## 2. Data Preprocessing

Dữ liệu gốc có nhiều cột bị đọc sai kiểu (numeric bị lưu dạng string), giá trị NULL rải rác ở nhiều cột, và các biến phân loại dạng chữ chưa thể đưa thẳng vào mô hình. Phần này xử lý lần lượt 3 việc: giảm dung lượng bộ nhớ, làm sạch NULL theo từng cột, và mã hoá (Label Encoding) các biến phân loại.

### 2.1. Tối ưu kiểu dữ liệu để giảm dung lượng bộ nhớ

`train_ver2.csv` có khoảng 13 triệu dòng, nếu giữ nguyên kiểu mặc định (`int64`/`float64`) sẽ rất tốn RAM. Hàm `reduce_memory_usage` duyệt qua từng cột số, kiểm tra khoảng giá trị min–max rồi ép về kiểu nhỏ nhất đủ chứa (ví dụ `int64` → `int8`/`int16`/`int32` nếu phù hợp) mà không làm mất thông tin.

In [ ]:
def reduce_memory_usage(df, verbose=True):
    numerics = ["int8", "int16", "int32", "int64", "float16", "float32", "float64"]
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if (
                    c_min > np.finfo(np.float16).min
                    and c_max < np.finfo(np.float16).max
                ):
                    df[col] = df[col].astype(np.float16)
                elif (
                    c_min > np.finfo(np.float32).min
                    and c_max < np.finfo(np.float32).max
                ):
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    if verbose:
        print(
            "Mem. usage decreased to {:.2f} Mb ({:.1f}% reduction)".format(
                end_mem, 100 * (start_mem - end_mem) / start_mem
            )
        )
    return df

In [ ]:
# Trước khi giảm
before_df = df.memory_usage().sum() / 1024 ** 2  # Đổi sang MB
before_df_test = df_test.memory_usage().sum() / 1024 ** 2

# Giảm bộ nhớ
df = reduce_memory_usage(df)
df_test = reduce_memory_usage(df_test)

# Sau khi giảm
after_df = df.memory_usage().sum() / 1024 ** 2  
after_df_test = df_test.memory_usage().sum() / 1024 ** 2

print("So sanh truoc va sau khi giam")
print("Tap train: ", before_df, " - ",after_df)
print("Tap test: ", before_df_test, " - ",after_df_test)

### 2.2. Xử lý giá trị NULL

Mỗi cột có đặc thù riêng nên cần chiến lược làm sạch khác nhau — cột thì suy ra từ cột khác (`antiguedad`, `days`), cột thì impute theo nhóm (`renta`), cột thì có quy tắc nghiệp vụ rõ ràng để fill giá trị mặc định (`indrel`, `indfall`, ...). Dưới đây xử lý lần lượt từng cột có NULL/giá trị bẩn.

#### `age` — Tuổi khách hàng

Ép kiểu số (giá trị lỗi → `NaN`), sau đó xử lý outlier: tuổi < 18 được thay bằng tuổi trung bình nhóm 18–30, tuổi > 100 được thay bằng tuổi trung bình nhóm 30–100. Phần còn thiếu (`NaN`) được fill bằng tuổi trung bình toàn bộ, rồi ép về kiểu `int`.

In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

n_young = int((df["age"] < 18).sum())
n_old = int((df["age"] > 100).sum())
print(f"Outliers: {n_young} tuổi < 18, {n_old} tuổi > 100")

mean_young = df.loc[(df.age >= 18) & (df.age <= 30), "age"].mean()
mean_mid = df.loc[(df.age >= 30) & (df.age <= 100), "age"].mean()

df.loc[df.age < 18, "age"] = mean_young
df.loc[df.age > 100, "age"] = mean_mid
df["age"] = df["age"].fillna(df["age"].mean())
df["age"] = df["age"].astype(int)

#### `antiguedad` — Thâm niên khách hàng (số tháng)

Chuyển `fecha_dato` (ngày snapshot) và `fecha_alta` (ngày trở thành khách hàng) sang kiểu ngày tháng, rồi tính lại thâm niên bằng chênh lệch số tháng giữa hai mốc này — đáng tin cậy hơn giá trị `antiguedad` gốc (vốn có lẫn dữ liệu bẩn/không nhất quán).

In [ ]:
df["fecha_dato"] = pd.to_datetime(df["fecha_dato"], errors="coerce")
df["fecha_alta"] = pd.to_datetime(df["fecha_alta"], errors="coerce")

df["antiguedad"] = pd.to_numeric(df["antiguedad"], errors="coerce")

antiguedad_calc = (
    (df["fecha_dato"].dt.year - df["fecha_alta"].dt.year) * 12
    + (df["fecha_dato"].dt.month - df["fecha_alta"].dt.month)
)

In [ ]:
print(f"fecha_dato null sau convert: {df['fecha_dato'].isnull().sum()}")
print(f"fecha_alta null sau convert: {df['fecha_alta'].isnull().sum()}")

#### `renta` — Thu nhập hộ gia đình

Đây là cột thiếu nhiều nhất. Chiến lược: tính thu nhập trung bình theo từng nhóm `(nomprov, segmento)` (tỉnh × phân khúc khách hàng), rồi merge (thay vì `apply` từng dòng — merge theo nhóm nhanh hơn nhiều trên dữ liệu 13 triệu dòng) để fill giá trị thiếu. Nếu cả nhóm cũng không có dữ liệu, fallback về trung bình toàn bộ tập.

In [ ]:
df['renta'] = pd.to_numeric(df['renta'], errors='coerce')
df_test['renta'] = pd.to_numeric(df_test['renta'], errors='coerce')

# Tính mean theo nhóm
mean_gross_classified_df = df.groupby(['nomprov', 'segmento'])['renta'].mean().reset_index()
mean_gross_classified_df.rename(columns={'renta': 'mean_gross_classified'}, inplace=True)

overall_renta = df['renta'].mean()
mean_gross_classified_df['mean_gross_classified'] = mean_gross_classified_df['mean_gross_classified'].fillna(overall_renta)

# Merge thay vì apply — vectorized, nhanh hơn rất nhiều
def fill_renta_vectorized(data):
    data = data.merge(mean_gross_classified_df, on=['nomprov', 'segmento'], how='left')
    data['renta'] = data['renta'].fillna(data['mean_gross_classified'])
    data['renta'] = data['renta'].fillna(overall_renta)  # fallback cho case nomprov/segmento cũng null
    data.drop(columns=['mean_gross_classified'], inplace=True)
    return data

df = fill_renta_vectorized(df)
df_test = fill_renta_vectorized(df_test)

#### `ind_nomina_ult1`, `ind_nom_pens_ult1` — Cờ sở hữu sản phẩm lương/hưu trí

Sắp xếp dữ liệu theo `(ncodpers, fecha_dato)` rồi forward-fill (`ffill`) theo từng khách hàng — giả định trạng thái sản phẩm giữ nguyên từ tháng gần nhất nếu bị thiếu. Những trường hợp vẫn còn thiếu sau ffill (khách hàng mới, chưa có lịch sử) được fill bằng 0 (chưa sở hữu).

In [ ]:
df = df.sort_values(["ncodpers", "fecha_dato"])

for col in ["ind_nomina_ult1", "ind_nom_pens_ult1"]:
    df[col] = df.groupby("ncodpers")[col].ffill()
    df[col] = df[col].fillna(0)

#### `indrel_1mes` — Loại khách hàng tại đầu tháng

Cột này bị lưu lẫn lộn nhiều kiểu dữ liệu cho cùng một giá trị (ví dụ `1`, `1.0`, `"1"`, `"1.0"` đều nghĩa là *primary*). Dùng `map_dict` để chuẩn hoá tất cả về một dạng string thống nhất (`"1"`, `"2"`, `"3"`, `"4"`, `"P"`), giá trị thiếu được fill là `"P"` (potential customer), sau đó ép về kiểu `category`.

In [ ]:
map_dict = { 1.0  : "1",
            "1.0" : "1",
            "1"   : "1",
            "3.0" : "3",
            "P"   : "P",
            3.0   : "3",
            2.0   : "2",
            "3"   : "3",
            "2.0" : "2",
            "4.0" : "4",
            "4"   : "4",
            "2"   : "2"}

df['indrel_1mes'] = df['indrel_1mes'].fillna("P")
df['indrel_1mes'] = df['indrel_1mes'].apply(lambda x: map_dict.get(x, x))
df['indrel_1mes'] = df['indrel_1mes'].astype("category")

df_test['indrel_1mes'] = df_test['indrel_1mes'].fillna("P")
df_test['indrel_1mes'] = df_test['indrel_1mes'].apply(lambda x: map_dict.get(x, x))
df_test['indrel_1mes'] = df_test['indrel_1mes'].astype("category")

#### `ind_nuevo` — Cờ khách hàng mới

Với các dòng bị thiếu, kiểm tra số tháng hoạt động (`groupby("ncodpers").size()`) của nhóm khách hàng đó: nếu số tháng active thấp thì khả năng cao đây thực sự là khách hàng mới → fill giá trị `1`.

In [ ]:
df["ind_nuevo"] = pd.to_numeric(df["ind_nuevo"], errors="coerce")
n_missing = df["ind_nuevo"].isnull().sum()
print(f"Missing: {n_missing}")

if n_missing > 0:
    months_active = df.loc[df["ind_nuevo"].isnull(), :].groupby("ncodpers").size()
    print(f"Max số tháng active trong nhóm thiếu dữ liệu: {months_active.max()}")
    # Số tháng active thấp -> đúng là khách hàng mới
    df.loc[df["ind_nuevo"].isnull(), "ind_nuevo"] = 1

#### `indrel` — Cờ khách hàng chính (primary)

Ép kiểu `float` và fill giá trị thiếu bằng `1.0` (mặc định là khách hàng chính, do đại đa số bản ghi thuộc nhóm này).

In [ ]:
df['indrel'] = df['indrel'].astype(float)
df_test['indrel'] = df_test['indrel'].astype(float)
df['indrel'] = df['indrel'].fillna(1.0)
df_test['indrel'] = df_test['indrel'].fillna(1.0)

#### `ind_actividad_cliente` — Cờ hoạt động của khách hàng

Chỉ cần ép về kiểu `float` cho đồng nhất — cột này không có NULL cần xử lý thêm.

In [ ]:
df['ind_actividad_cliente'] = df['ind_actividad_cliente'].astype(float)
df_test['ind_actividad_cliente'] = df_test['ind_actividad_cliente'].astype(float)

#### `indfall` — Cờ khách hàng đã mất

Fill giá trị thiếu bằng `"N"` (mặc định: còn sống), vì đây là trường hợp phổ biến nhất khi thiếu dữ liệu.

In [ ]:
df['indfall'] = df['indfall'].fillna("N")
df_test['indfall'] = df_test['indfall'].fillna("N")

### 2.3. Mã hoá biến phân loại (Label Encoding)

Fit `LabelEncoder` cho từng cột phân loại **trên tập train**, sau đó áp dụng ánh xạ đó lên tập test. Vì tập test có thể xuất hiện giá trị chưa từng thấy trong train (encoder sẽ báo lỗi nếu dùng `transform` trực tiếp), nên ở đây map thủ công qua `dict` và gán `-1` cho các giá trị lạ (unseen category).

In [ ]:
from sklearn.preprocessing import LabelEncoder

cols = ['ind_empleado','pais_residencia', 'sexo','ind_nuevo','indrel_1mes','tiprel_1mes', 'indresi', 
        'indext', 'conyuemp', 'canal_entrada','indfall', 'nomprov','segmento','indrel','ind_actividad_cliente']

label_encoders = {}  # lưu các bộ encoder theo cột

for col in cols:
    le = LabelEncoder()
    # fit trên df[col], chuyển giá trị sang dạng string để tránh lỗi nếu có NA hoặc khác kiểu
    df[col] = df[col].astype(str)
    df_test[col] = df_test[col].astype(str)
    
    le.fit(df[col])
    df[col] = le.transform(df[col])
    
    # áp dụng encoder lên df_test
    # với các giá trị không thuộc tập train, transform sẽ lỗi nên ta xử lý thủ công
    # cách đơn giản: map theo dict encoder, giá trị không tìm thấy cho -1 (hoặc 9999 tùy bạn)
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    df_test[col] = df_test[col].map(mapping).fillna(-1).astype(int)
    
    label_encoders[col] = le

Kiểm tra nhanh các giá trị đã mã hoá trên tập **train**, đảm bảo không có gì bất thường:

In [ ]:
for col in cols:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique encoded values ({len(unique_vals)}): {sorted(unique_vals)}")
    print("------------------------------------------------------")

Kiểm tra tương tự trên tập **test** — chú ý các giá trị `-1` chính là category lạ chỉ xuất hiện ở test:

In [ ]:
for col in cols:
    unique_vals = df_test[col].unique()
    print(f"Column: {col}")
    print(f"Unique encoded values ({len(unique_vals)}): {sorted(unique_vals)}")
    print("------------------------------------------------------")

> **Lưu ý:** `ind_nuevo`, `indrel`, `ind_actividad_cliente` tuy bản chất là numeric/binary nhưng vẫn được đưa vào danh sách mã hoá cùng các biến category — vì trước đó đã bị ép qua `str`/`float` không đồng nhất, nên xử lý qua `LabelEncoder` để đảm bảo giá trị rời rạc và nhất quán giữa train/test.

## 3. Feature Engineering

Loại bỏ các đặc trưng/sản phẩm gây nhiễu hoặc không có tín hiệu, tạo thêm đặc trưng thời gian và đặc trưng biến động hành vi, sau đó xác định danh sách 24 sản phẩm mục tiêu sẽ dùng để xây tập nhãn.

### 3.1. Loại bỏ các đặc trưng không cần thiết

- `tipodom`: hầu như chỉ có một giá trị duy nhất → không mang thông tin phân biệt.
- `cod_prov`: trùng lặp thông tin với `nomprov` (mã tỉnh vs. tên tỉnh).
- `ult_fec_cli_1t`: thiếu gần như toàn bộ (chỉ có giá trị khi khách hàng vừa mất trạng thái *primary*).
- `conyuemp`: thiếu gần như toàn bộ.

In [ ]:
df.drop(columns=['tipodom', 'cod_prov', 'ult_fec_cli_1t', 'conyuemp'], inplace=True)
df_test.drop(columns=['tipodom', 'cod_prov', 'ult_fec_cli_1t', 'conyuemp'], inplace=True)

### 3.2. Loại bỏ các sản phẩm hiếm khi được mua mới

`ind_ahor_fin_ult1`, `ind_aval_fin_ult1`, `ind_cder_fin_ult1`, `ind_deme_fin_ult1` gần như không có khách hàng mua mới trong dữ liệu train → giữ lại chỉ làm tăng nhiễu và mất cân bằng lớp mà không cải thiện được model.

In [ ]:
df.drop(columns=['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cder_fin_ult1', 'ind_deme_fin_ult1'],inplace = True)

### 3.3. Thay `fecha_alta` bằng `days` (số ngày đã là khách hàng)

Thay vì giữ nguyên ngày tháng dạng datetime (mô hình cây không dùng trực tiếp được), tính số ngày chênh lệch giữa `fecha_dato` (ngày snapshot) và `fecha_alta` (ngày gia nhập) thành một đặc trưng số duy nhất — `days` — rồi bỏ cột `fecha_alta` gốc. Giá trị `days` bị thiếu (do `fecha_alta` gốc lỗi) được fill bằng giá trị trung bình.

In [ ]:
df['fecha_dato'] = pd.to_datetime(df.fecha_dato)
df['fecha_alta'] = pd.to_datetime(df.fecha_alta)
df_test['fecha_dato'] = pd.to_datetime(df_test.fecha_dato)
df_test['fecha_alta'] = pd.to_datetime(df_test.fecha_alta)
df_days_column = (df['fecha_dato'] - df['fecha_alta']).dt.days
df_test_days_column =  (df_test['fecha_dato'] - df_test['fecha_alta']).dt.days

#generate new column 'days' from 
df.insert(loc=6, column='days', value=df_days_column)
df_test.insert(loc=6, column='days', value=df_test_days_column)

#Drop the 'fetch_alta' column
df.drop(columns=['fecha_alta'],inplace = True)
df_test.drop(columns=['fecha_alta'],inplace = True)

In [ ]:
df.days.isnull().any()

In [ ]:
df['days'] = df['days'].fillna(df['days'].mean())
df_test['days'] = df_test['days'].fillna(df['days'].mean())

### 3.4. Tạo đặc trưng biến động hành vi (behavior change flags)

Với các cột phản ánh hành vi/trạng thái khách hàng tại một thời điểm (`segmento`, `ind_actividad_cliente`, `tiprel_1mes`), tạo thêm cột lấy giá trị tháng trước (`shift(1)` theo từng `ncodpers`) và cờ `_changed` đánh dấu tháng đó có thay đổi so với tháng liền trước hay không. Tháng đầu tiên của mỗi khách hàng (không có tháng trước để so sánh) được coi là *không thay đổi*.

In [ ]:
df = df.sort_values(["ncodpers", "fecha_dato"])

behavior_cols = [c for c in ["segmento", "ind_actividad_cliente", "tiprel_1mes"] if c in df.columns]
for col in behavior_cols:
    lag_col = f"{col}_lag_1"
    df[lag_col] = df.groupby("ncodpers")[col].shift(1)
    df[f"{col}_changed"] = (df[col] != df[lag_col]).astype(int)
    # Tháng đầu tiên của mỗi khách hàng không có lag -> không tính là "changed"
    df.loc[df[lag_col].isnull(), f"{col}_changed"] = 0

df[[f"{c}_changed" for c in behavior_cols]].mean()

### 3.5. Xác định danh sách 24 sản phẩm mục tiêu (`target_cols`)

Lọc lại danh sách toàn bộ sản phẩm gốc của cuộc thi, loại đi 4 sản phẩm đã drop ở bước 3.2, kết quả còn lại là `target_cols` — danh sách sản phẩm sẽ được dùng làm nhãn dự đoán.

In [ ]:
ALL_ORIGINAL_PRODUCTS = [
    'ind_ahor_fin_ult1','ind_aval_fin_ult1','ind_cco_fin_ult1','ind_cder_fin_ult1',
    'ind_cno_fin_ult1','ind_ctju_fin_ult1','ind_ctma_fin_ult1','ind_ctop_fin_ult1',
    'ind_ctpp_fin_ult1','ind_deco_fin_ult1','ind_deme_fin_ult1','ind_dela_fin_ult1',
    'ind_ecue_fin_ult1','ind_fond_fin_ult1','ind_hip_fin_ult1','ind_plan_fin_ult1',
    'ind_pres_fin_ult1','ind_reca_fin_ult1','ind_tjcr_fin_ult1','ind_valo_fin_ult1',
    'ind_viv_fin_ult1','ind_nomina_ult1','ind_nom_pens_ult1','ind_recibo_ult1'
]

dropped = [c for c in ALL_ORIGINAL_PRODUCTS if c not in df.columns]
print("Các cột đã bị drop:", dropped)

target_cols = np.array([c for c in ALL_ORIGINAL_PRODUCTS if c in df.columns])
print(len(target_cols))
print(target_cols)

### 3.6. Xác định khách hàng có mua thêm sản phẩm mới (T5 → T6/2015)

So sánh snapshot tháng 05/2015 và 06/2015 của cùng một khách hàng: nếu một cột sản phẩm chuyển từ `1` (T5) xuống không còn `1` theo phép trừ (`cust5 - cust6 == -1`, tức từ 0 lên 1 hoặc khách mua mới), khách hàng đó được xem là có mua thêm sản phẩm. Tách riêng hai nhóm:
- `cust5_buyin6`: khách hàng đã tồn tại ở T5, có mua thêm sản phẩm ở T6.
- `cust6_newbuyin6`: khách hàng hoàn toàn mới, chỉ xuất hiện lần đầu ở T6.

In [ ]:
cust5_2015 = df[df['fecha_dato'] == '2015-05-28'].set_index('ncodpers')[target_cols]
id_cust5_2015 = cust5_2015.index.to_numpy()
cust6_in52015 = df[(df['fecha_dato'] == '2015-06-28') & (df['ncodpers'].isin(cust5_2015.index))] \
               .set_index('ncodpers')[target_cols]
subtract_56 = (cust5_2015 - cust6_in52015)
q = (subtract_56[target_cols] == -1).sum(1)
id_cust5_buyin6 = q[q > 0].index

cust5_buyin6 = df[(df['fecha_dato'] == '2015-06-28') & (df['ncodpers'].isin(id_cust5_buyin6))]
cust6_newbuyin6 = df[(df['fecha_dato'] == '2015-06-28') & (~df['ncodpers'].isin(id_cust5_2015))]

### 3.7. Gộp thành `train_total` theo dạng long-format (mỗi dòng ứng với 1 sản phẩm)

Với mỗi sản phẩm trong `target_cols`, lọc ra các khách hàng đã mua thêm đúng sản phẩm đó (từ cả 2 nhóm ở bước 3.6), gán nhãn `target = t` (chỉ số thứ tự của sản phẩm), rồi nối tất cả lại thành `train_total`. Cách làm này biến bài toán multi-label (24 cột nhị phân) thành bài toán **multi-class 1 nhãn duy nhất** (dự đoán sản phẩm nào được mua thêm), đơn giản hoá việc huấn luyện.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

train_total = pd.DataFrame()
t=0
for i in tqdm(target_cols):
    train = cust5_buyin6[cust5_buyin6['ncodpers'].isin(subtract_56[subtract_56[i] == -1].index)]
    train2 = cust6_newbuyin6[cust6_newbuyin6[i] == 1]
    train.drop(columns=target_cols,inplace = True)
    train2.drop(columns=target_cols,inplace = True)
    train['target'] = t
    train2['target'] = t

    train = pd.concat([train, train2], ignore_index=True, sort=False)
    train_total = pd.concat([train_total, train], ignore_index=True, sort=False)
    t+=1

del train
del train2
warnings.filterwarnings('default')

### 3.8. Ghép `train_total` với `df_test` thành `df_total`

Gộp tập train đã tạo nhãn với tập test gốc (test không có cột `target`) thành một `DataFrame` duy nhất, tiện cho các bước xử lý đặc trưng chung phía sau.

In [ ]:
df_total = pd.concat([train_total, df_test], ignore_index=True, sort=False)
df_total

## 4. Chuẩn bị tập Train / Test cho mô hình

Từ `df_total`, tách lại thành `X_train`/`y_train` (các dòng có `target`) và `X_test` (các dòng `target` là `NaN`, chính là tập test gốc), đồng thời bổ sung đặc trưng lag lịch sử sản phẩm cho từng khách hàng.

### 4.1. Tạo tập Train

Lọc các dòng có `target` khác `NaN`, tách `y_train`, loại bỏ cột `ncodpers`/`fecha_dato` khỏi đặc trưng đầu vào.

In [ ]:
X_train = df_total[~df_total['target'].isnull()]
y_train = X_train['target'].astype(int)
X_train.drop(columns=['target'],inplace = True)

train_cust_ids = X_train['ncodpers']
X_train.drop(columns=['fecha_dato','ncodpers'],inplace = True)

X_train = X_train.values.tolist()

### 4.2. Tạo tập Test

Lọc các dòng có `target` là `NaN` (đây chính là tập test gốc cần dự đoán), xử lý tương tự tập train.

In [ ]:
test_cust_ids = df_total[df_total['target'].isnull()]['ncodpers'].values
X_test = df_total[df_total['target'].isnull()]
X_test.drop(columns=['target'],inplace=True) 
X_test.drop(columns=['fecha_dato','ncodpers'],inplace = True)
X_test = X_test.values.tolist()

### 4.3. Sinh 5 đặc trưng lag lịch sử sản phẩm cho mỗi khách hàng

Khối này thực hiện 4 việc liên tiếp:
1. **`train_lags`**: với mỗi khách hàng trong `train_total`, lấy lịch sử 24 cột sản phẩm ở các tháng trước tháng 06/2015, lưu thành list các vector (mỗi vector = trạng thái sản phẩm 1 tháng).
2. **`test_lags`**: tương tự nhưng lấy lịch sử từ 01/2016 đến trước 06/2016 cho khách hàng trong tập test.
3. Với mỗi khách hàng ở `X_train`, lấy 5 vector lịch sử gần nhất (`lag_1`…`lag_5`, thiếu thì fill toàn `0`) và nối (`extend`) vào cuối vector đặc trưng hiện có.
4. Lặp lại quy trình tương tự cho `X_test`, kết quả là `X_test_new` (danh sách đặc trưng đã có thêm lag).

Kết quả cuối: `X_train` là `np.array` gồm đặc trưng gốc + 5×24 đặc trưng lag; `X_test` là list tương ứng cho tập test.

In [ ]:
# Generating 5-lag features for every product

#Train lag features
temp1 = df[(df['fecha_dato'] < '2015-06-28') & (df['ncodpers'].isin(train_total['ncodpers']))]
temp1.drop(columns = ['ind_empleado','pais_residencia','sexo','age','days','ind_nuevo','antiguedad','indrel','indrel_1mes','tiprel_1mes','indresi','indext','canal_entrada','indfall','nomprov','ind_actividad_cliente','renta','segmento'],inplace = True)

train_lags = {}
for i in tqdm(temp1.itertuples()):
    if i[2] not in train_lags.keys():
        train_lags[i[2]] = []
    train_lags[i[2]].append(np.array(i[3:]).astype(int))


#Test lag features
temp2 = df[(df['fecha_dato'] < '2016-06-28') & (df['fecha_dato'] >= '2016-01-28') & (df['ncodpers'].isin(df_test['ncodpers']))]
temp2.drop(columns = ['ind_empleado','pais_residencia','sexo','age','days','ind_nuevo','antiguedad','indrel','indrel_1mes','tiprel_1mes','indresi','indext','canal_entrada','indfall','nomprov','ind_actividad_cliente','renta','segmento'],inplace = True)

test_lags = {}
for i in tqdm(temp2.itertuples()):
    if i[2] not in test_lags.keys():
        test_lags[i[2]] = []
    test_lags[i[2]].append(np.array(i[3:]).astype(int))


#Creating the final train dataset
X_train = df_total[~df_total['target'].isnull()]
y_train = X_train['target'].astype(int)
X_train.drop(columns=['target'],inplace = True)

train_cust_ids = X_train['ncodpers']
X_train.drop(columns=['fecha_dato','ncodpers'],inplace = True)

X_train = X_train.values.tolist()

#Adding the lag variables to the train dataset
N_LAG_COLS = temp1.shape[1] - 2 
k=0
for i in tqdm(train_cust_ids):
    l = train_lags.get(i,[[0]*N_LAG_COLS])

    try:
        lag_1 = list(l[-1])
    except:
        lag_1 = [0]*N_LAG_COLS
    try:
        lag_2 = list(l[-2])
    except:
        lag_2 = [0]*N_LAG_COLS
    try:
        lag_3 = list(l[-3])
    except:
        lag_3 = [0]*N_LAG_COLS
    try:
        lag_4 = list(l[-4])
    except:
        lag_4 = [0]*N_LAG_COLS
    try:
        lag_5 = list(l[-5])
    except:
        lag_5 = [0]*N_LAG_COLS
    
    X_train[k].extend(lag_1+lag_5+lag_4+lag_3+lag_2)
    k+=1  

X_train = np.array(X_train)

##Creating the final test dataset
test_cust_ids = df_total[df_total['target'].isnull()]['ncodpers'].values
X_test = df_total[df_total['target'].isnull()]
X_test.drop(columns=['target'],inplace=True) 
X_test.drop(columns=['fecha_dato','ncodpers'],inplace = True)
X_test = X_test.values.tolist()

X_test_new = []

#Adding the lag variables to the test dataset
k=0
for i in tqdm(test_cust_ids):
    l = test_lags.get(i,[[0]*N_LAG_COLS])

    try:
        lag_1 = list(l[-1])
    except:
        lag_1 = [0]*N_LAG_COLS
    try:
        lag_2 = list(l[-2])
    except:
        lag_2 = [0]*N_LAG_COLS
    try:
        lag_3 = list(l[-3])
    except:
        lag_3 = [0]*N_LAG_COLS
    try:
        lag_4 = list(l[-4])
    except:
        lag_4 = [0]*N_LAG_COLS
    try:
        lag_5 = list(l[-5])
    except:
        lag_5 = [0]*N_LAG_COLS
    
    X_test_new.append(np.array(X_test[k] + lag_1 + lag_5 + lag_4 +lag_3 + lag_2))
    k+=1


X_test = X_test_new


In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test length:", len(X_test))

## 5. Huấn luyện mô hình & Dự đoán

Bài toán được đưa về dạng multi-class (24 lớp = 24 sản phẩm). Quy trình: dò siêu tham số nhanh bằng `RandomizedSearchCV`, sau đó train mô hình XGBoost cuối cùng với tham số tốt nhất, dự đoán xác suất cho từng sản phẩm và chọn ra **top 7 sản phẩm khách hàng chưa sở hữu** có xác suất cao nhất (đúng theo tiêu chí chấm điểm MAP@7 của cuộc thi).

### 5.1. Dò siêu tham số

Khai báo lưới tham số cho cả 4 thuật toán ứng viên (XGBoost, Random Forest, CatBoost, LightGBM), nhưng ở bước này chỉ chạy thử `RandomizedSearchCV` với **LightGBM** — thuật toán nhẹ và nhanh nhất trong 4 lựa chọn — để kiểm tra nhanh cấu hình và pipeline trước khi mở rộng ra các thuật toán còn lại.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# Tham số cho từng thuật toán
params_xgb = {
    'learning_rate': [0.01, 0.03, 0.1, 0.2],
    'max_depth': [3, 5, 8],
    'n_estimators': [10, 50],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
    'min_child_weight': [1, 3, 5, 7, 10, 14]
}

params_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

params_cat = {
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [100, 200],
    'l2_leaf_reg': [1, 3, 5, 7, 9]
}

params_lgbm = {
    'num_leaves': [31, 50, 70],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200],
    'min_child_samples': [10, 20, 30]
}

# Chỉ chạy LGBM trước — nhẹ và nhanh nhất trong 4 model
clf = RandomizedSearchCV(
    LGBMClassifier(objective='multiclass', n_jobs=1),
    params_lgbm,
    n_iter=10,           # giảm từ 15 xuống 10 để chạy thử nhanh trước
    scoring='neg_log_loss',  # nhẹ hơn roc_auc_ovo
    cv=3,
    n_jobs=-1,
    verbose=10,
    error_score='raise'
)
clf.fit(X_train, y_train)
print(f"Best params: {clf.best_params_}")

In [ ]:
clf.best_params_

### 5.2. Huấn luyện XGBoost với tham số tốt nhất & Dự đoán Top-7 sản phẩm

- Encode `y_train` bằng `LabelEncoder` (`le`) để dùng cho `objective='multi:softprob'`.
- Train `XGBClassifier` với bộ tham số đã chọn, dự đoán xác suất cho từng lớp (`predict_proba`) trên `X_test`.
- Với mỗi khách hàng test, xác định các sản phẩm **chưa sở hữu** tính đến snapshot gần nhất (05/2016) — chỉ những sản phẩm này mới hợp lệ để đề xuất mua thêm.
- Lọc xác suất dự đoán chỉ trên các sản phẩm hợp lệ đó, sắp xếp giảm dần và lấy **7 sản phẩm có xác suất cao nhất**.

In [ ]:
#best hyperparameters
from xgboost.sklearn import XGBClassifier

y_train_encoded = le.fit_transform(y_train)
xgb = XGBClassifier(objective = 'multi:softprob',eval_metric = 'mlogloss',max_depth=3,n_estimators=50,learning_rate=0.03,colsample_bytree=0.8,subsample=0.9,min_child_weight = 1)
xgb.fit(X_train, y_train_encoded)

y_pred = xgb.predict_proba(X_test)

#Finding the new products to be predicted by test users
test_users = {}
for i in tqdm(df[(df['fecha_dato'] == '2016-05-28') & (df['ncodpers'].isin(df_test['ncodpers'].to_numpy()))].iloc):
    test_users[i['ncodpers']] = np.where(i['ind_cco_fin_ult1':'ind_recibo_ult1'] == 0)[0]


label_to_col = {label: idx for idx, label in enumerate(le.classes_)}

test_user_ratings = {}
test_users_valid = {}
k = 0
for i in tqdm(test_cust_ids):
    original_zero_idx = test_users[i]
    valid_idx = [idx for idx in original_zero_idx if idx in label_to_col]
    cols_in_pred = [label_to_col[idx] for idx in valid_idx]
    test_user_ratings[i] = y_pred[k][cols_in_pred]
    test_users_valid[i] = valid_idx   # dùng cái này thay cho test_users[i] ở bước sau
    k += 1

final_preds = []
test_ids = []
for i in tqdm(test_user_ratings.keys()):
    sorted_idx = np.argsort(test_user_ratings[i])[::-1][:7]
    top_seven = [test_users_valid[i][j] for j in sorted_idx]
    final_preds.append(" ".join(target_cols[np.array(top_seven)]))
    test_ids.append(i)

### 5.3. Xuất file submission

Ghép `ncodpers` với chuỗi 7 sản phẩm dự đoán (`added_products`), sắp xếp theo `ncodpers` và lưu ra `sub.csv` đúng định dạng nộp bài của cuộc thi.

In [ ]:
xgb_submission = pd.DataFrame({'ncodpers':test_ids, 'added_products':final_preds})
xgb_submission = xgb_submission.sort_values('ncodpers')
xgb_submission.to_csv('sub.csv', index=False)